# 对照已有的本地检索与联网研究

本页学习目标：

- 对照真实记录，理解检索、读取、回答和引用检查。
- 区分本地预检索与模型主动搜索网页。
- 判断有链接的回答是否真正完成研究问题。

本页介绍 2026-10-20 的两次真实实验，使用 CPU 上的 Qwen3-0.6B。重新运行模型的分支默认关闭。

[返回课程入口](../README.md) · 理解基本循环后再看。


## 1. 本地实验是怎样运行的

1. 程序读取本地教学文档，用 all-MiniLM-L6-v2 生成 384 维向量。
2. 程序检索相关片段，把文本和来源编号放进模型请求。
3. 模型回答 Agent 与 workflow 的区别。
4. 程序检查引用，再直接检查文件、数据库和算术工具。

模型的回答说明：Agent 根据工具反馈行动，workflow 的步骤通常由程序预先指定。两句话分别引用实际提供的 `L1`、`L2` 片段。

文件检查取得了文档正文；数据库检查返回了关于 Agent 的记录；受限代码执行了 `12 * 34 + 2`，返回 410。后三项由程序直接调用，没有模型自主选择工具的过程。


## 2. 联网研究是怎样运行的

任务是解释“工具调用怎样让外部执行结果进入下一轮模型输入”。程序提供 `search_web` 和 `read_page`，模型提出调用，程序实际访问网页。

本实现的搜索范围是官方站点的 sitemap，再按 URL 中的词排序，并非通用搜索引擎。阅读时优先取得官方 Markdown 正文，减少导航内容。

实际运行有两次工具调用：搜索官方工具调用资料，然后读取正文。成功读取的来源是 Ollama 的 tool-calling 文档，编号为 `L1`。

这里可以确认搜索和读取实际发生过，但只涉及一个较窄的官方资料任务，不能代表任意互联网研究能力。


## 3. 真实报告为什么仍然不够好

模型最终回答的意思是：

> 工具调用循环已成功执行，已通过来源 L1 获取第一页；L1 提供工具调用官方文档，并包含解释过程的文本。

这是对实际英文输出的概述。回答有合法来源编号，但主要在说“找到了文档”，没有清楚解释：

1. 模型如何生成工具请求。
2. 程序怎样执行请求。
3. 工具结果怎样放进消息历史。
4. 下一轮模型怎样利用结果。

程序的引用检查通过，`status` 为 `completed`，但研究问题没有得到充分回答。因此，编号合法和循环结束之外，还需要检查答案内容。不能把原始报告改成更好的版本，再冒充模型当时已经答好。


## 4. 一份研究回答应该包含什么

以本页问题为例，至少应解释以下过程：

```text
用户问题与工具说明
    ↓
模型生成工具请求
    ↓
程序检查并执行工具
    ↓
工具返回正文或错误
    ↓
程序把结果加入消息历史
    ↓
模型据此选择下一动作或回答
```

这段是本页的教学说明，不是替换后的实验成绩。真正的研究回答还需要把相应结论绑定到实际读过的原文，并说明没有找到依据的部分。


## 5. 资料打不开或检索为空时怎么办

| 情况 | 应该保留什么 | 可以怎样处理 |
| --- | --- | --- |
| 找不到候选 | 查询和空列表 | 换词或说明没有找到资料 |
| 正文请求失败 | 实际 URL 和错误 | 换允许的读取渠道或其他来源 |
| 只有动态页面外壳 | 实际取得的文本 | 在允许时使用浏览器，否则说明正文缺失 |
| 只有搜索摘要 | 候选状态 | 继续读取，不能声称读过正文 |
| 有正文但没回答问题 | 原始回答和对应资料 | 判为回答不充分，再改进生成和检查 |

这些情况需要分别记录。工具失败不能写成模型事实判断失败；引用格式通过也不能消除内容缺失。


## 6. 练习与自查

对一份回答逐条检查：问题的每一部分是否被回答？每项结论引用了什么？正文是否成功读取？模型实际看到了多少？原文是否支持结论？

再区分每次读取是程序预先安排，还是模型请求触发。这样才能说明这个系统实际完成了哪部分工作。

下一页：[Self-Ask](04-extension-self-ask.ipynb)，或进入 [Agent Harness](../03-agent-harness/01-harness.ipynb)。


## 代码说明

`BoundedClient` 适配本地接口并记录预算，`run_local` 执行本地检索流程，`search` 和 `fetch` 取得官方资料，`run` 连接联网工具与模型。

默认代码检查对象转换，资料已经内嵌于研究助手 Notebook。`RUN_LIVE=False` 时不会重新执行向量生成、模型回答或联网研究。开启分支前需要配置实际可用的生成模型、向量模型和网络环境。


In [ ]:
from miniagent_course import ROOT, OUTPUT, load_lesson


In [ ]:
import time

In [ ]:
import argparse

import json

import re

import xml.etree.ElementTree as ET

from pathlib import Path

from urllib.parse import urlsplit

from urllib.request import Request, urlopen

from urllib.error import URLError, HTTPError

_dependency = load_lesson('01-agent-loop/02-live-model.ipynb')

chat = _dependency.chat

request = _dependency.request

model_manifest = _dependency.model_manifest

save = _dependency.save

parse_json = _dependency.parse_json

_dependency = load_lesson('03-agent-harness/01-harness.ipynb')

Harness = _dependency.Harness

OllamaModel = _dependency.OllamaModel

Tool = _dependency.Tool

schema = _dependency.schema

citation_errors = _dependency.citation_errors

<a id="impl-ResponseObject"></a>
<a id="impl-objectify"></a>

### 实现：ResponseObject, objectify


In [ ]:
import shutil
from types import SimpleNamespace

# 返回对象提供完整研究 Agent 所使用的属性访问与 model_dump 接口。
class ResponseObject(SimpleNamespace):
    def model_dump(self, **kwargs):
        def plain(value):
            if isinstance(value, ResponseObject):
                return {key: plain(item) for key, item in vars(value).items()}
            if isinstance(value, list):
                return [plain(item) for item in value]
            return value
        return plain(self)


def objectify(value):
    if isinstance(value, dict):
        return ResponseObject(**{key: objectify(item) for key, item in value.items()})
    if isinstance(value, list):
        return [objectify(item) for item in value]
    return value

<a id="impl-BoundedClient"></a>

### 实现：BoundedClient


In [ ]:
import copy

class BoundedClient:
    def __init__(self, model="qwen3:1.7b", embedding_model="qwen3-embedding:0.6b", timeout=120):
        self.model = model
        self.embedding_model = embedding_model
        self.deadline = time.monotonic() + timeout
        self.calls = []
        self.embedding_calls = []

    def remaining(self):
        remaining = self.deadline - time.monotonic()
        if remaining <= 0:
            raise TimeoutError("local research request budget exhausted")
        return remaining

    def chat(self, **kwargs):
        record = chat(kwargs["messages"], model=self.model, tools=kwargs.get("tools"),
                      max_output=512, timeout=self.remaining())
        raw = record["response"]
        raw["message"].setdefault("tool_calls", [])
        self.calls.append(copy.deepcopy(record))
        return objectify(raw)

    def embed(self, **kwargs):
        body = {**kwargs, "model": self.embedding_model}
        started = time.monotonic()
        raw = request("/api/embed", body, timeout=self.remaining())
        self.embedding_calls.append({"request": copy.deepcopy(body),
            "dimensions": [len(vector) for vector in raw["embeddings"]],
            "latency_ms": round((time.monotonic() - started) * 1000, 3)})
        return objectify(raw)

In [ ]:
# 仅核对接口转换，无模型调用。
sample = {"message": {"content": "example", "tool_calls": []}, "embeddings": [[1.0, 0.0]]}
converted = objectify(sample)
assert converted.message.content == "example"
assert converted.model_dump() == sample
print("对象接口转换通过；下面的真实检索实验需要开启 RUN_LIVE。")

<a id="impl-run_local"></a>

### 实现：run_local


In [ ]:
from miniagent_course import lesson_assets

def run_local(output, model="qwen3:1.7b", embedding_model="qwen3-embedding:0.6b"):
    lesson = load_lesson("02-tools-rag-memory/02-research-agent.ipynb")
    output = Path(output)
    workdir = output / "workspace"
    (workdir / "docs").mkdir(parents=True, exist_ok=True)
    for source in (lesson_assets("02-tools-rag-memory/02-research-agent.ipynb") / "docs").glob("*.md"):
        shutil.copyfile(source, workdir / "docs" / source.name)
    manifests = {"model": model_manifest(model), "embedding": model_manifest(embedding_model)}
    client = BoundedClient(model, embedding_model)
    old_config = lesson.MODEL, lesson.EMBED_MODEL, lesson.TIME_LIMIT
    lesson.MODEL, lesson.EMBED_MODEL, lesson.TIME_LIMIT = model, embedding_model, 120
    try:
        agent = lesson.ResearchAgent(base=workdir, llm=client, embed=client)
        hits = agent.rag.retrieve("Agent 与 workflow 怎样选择下一步？", min_score=None)
        answer = agent.run("Agent 和 Workflow 分别如何决定下一步？仅写两行，每行末尾引用本轮编号。",
                           local=True, repair_citations=True)
        validation = dict(agent.validation)
        answer_sources = dict(agent.sources)
        answer_trace = list(agent.trace)
        lesson.init_demo_db(workdir)
        agent._prepare_turn()
        tools = {
            "file": agent.execute_tool("read_file", {"path": "agent.md"}),
            "database": agent.execute_tool("query_database", {"query": "Agent"}),
            "code": agent.execute_tool("code_execute", {"code": "subtotal = 12 * 34\nsubtotal + 2"}),
        }
        index = agent.rag.ensure_index()
        dimension = len(index["chunks"][0]["embedding"]) if index["chunks"] else 0
        checks = {
            "retrieval_has_evidence": bool(hits),
            "real_embedding_requested": bool(client.embedding_calls) and dimension > 0,
            "answer_citations": bool(validation.get("passed")) and
                not answer.startswith(("任务未完成", "证据校验未通过")),
            "file_read": tools["file"].get("status") == "read" and bool(tools["file"].get("text")),
            "database_read": tools["database"].get("status") == "ok" and bool(tools["database"]["results"]),
            "restricted_code": tools["code"].get("result") == 410,
        }
        result = {**manifests, "retrieval": hits, "answer": answer, "validation": validation,
                  "answer_sources": answer_sources, "trace": answer_trace,
                  "requests": client.calls, "embedding_requests": client.embedding_calls,
                  "embedding_dimension": dimension, "tools": tools, "checks": checks,
                  "validation_passed": all(checks.values()),
                  "tool_test_mode": "direct program calls after the model answer"}
        save(output / "result.json", result)
        print(answer)
        print("真实向量维度：", dimension)
        print("分项验收：", checks)
        return result
    finally:
        lesson.MODEL, lesson.EMBED_MODEL, lesson.TIME_LIMIT = old_config

In [ ]:
RUN_LIVE = False
MODEL = "qwen3:1.7b"
EMBEDDING_MODEL = "qwen3-embedding:0.6b"
RUN_ID = str(time.time_ns())

In [ ]:
if RUN_LIVE:
    local_result = run_local(OUTPUT / "02-research" / RUN_ID / "local",
                             model=MODEL, embedding_model=EMBEDDING_MODEL)
    assert local_result["validation_passed"], local_result["checks"]
    print("本地检索、真实模型和三种实际工具的检查通过；继续执行下面的公开资料研究。")
else:
    print("本地真实模型、embedding、文件/数据库/算术组合实验未执行；不把接口转换测试当作真实实验完成。")

<a id="impl-fetch"></a>

### 实现：fetch


In [ ]:
def fetch(url):
    request = Request(url, headers={"User-Agent": "miniAgent-course-research/1.0"})
    for attempt in range(2):
        try:
            with urlopen(request, timeout=20) as response:
                data = response.read(500001)
            break
        except (TimeoutError, URLError) as error:
            if attempt or isinstance(error, HTTPError) and error.code not in {408, 429, 500, 502, 503, 504}:
                raise
    if len(data) > 500000:
        raise ValueError("page exceeds the lab size limit")
    return data.decode("utf-8", "replace")

<a id="impl-search"></a>

### 实现：search


In [ ]:
def search(query):
    if not query.strip() or len(query) > 300:
        raise ValueError("invalid query")
    # Search the official site's live public sitemap. This is a domain search,
    # not a cached URL list or a general Internet search engine.
    raw = fetch("https://docs.ollama.com/sitemap.xml")
    root = ET.fromstring(raw)
    query_terms = set(re.findall(r"[a-z]+", query.lower())) - {"official", "ollama", "documentation", "docs"}
    urls = [element.text for element in root.iter() if element.tag.endswith("loc") and element.text]
    scored = []
    for url in urls:
        parsed = urlsplit(url)
        if parsed.scheme == "https" and parsed.netloc == "docs.ollama.com":
            score = len(query_terms & set(re.findall(r"[a-z]+", parsed.path.lower())))
            if score:
                scored.append((score, url))
    results = [{"id": "L" + str(i + 1), "url": url,
                "search_backend": "live official sitemap, URL term ranking", "notice": "candidate, not yet read"}
               for i, (_, url) in enumerate(sorted(scored, reverse=True)[:3])]
    if not results:
        raise RuntimeError("search returned no allowlisted links; provider HTML may have changed")
    return results

<a id="impl-run"></a>

### 实现：run


In [ ]:
def run(output, model="qwen3:1.7b"):
    output.mkdir(parents=True, exist_ok=True)
    candidates, evidence = {}, {}
    def search_web(query):
        rows = search(query)
        candidates.update({row["id"]: row for row in rows})
        return rows
    def read_page(source_id):
        row = candidates[source_id]
        # Official Markdown avoids the site's large application/navigation HTML.
        markdown_url = row["url"].rstrip("/") + ".md"
        text = fetch(markdown_url)
        # Prefer the article content after its heading, reducing navigation noise.
        headings = ["Tool calling", "Generate a chat message", "Tool Calling"]
        positions = [text.find(h) for h in headings if h in text]
        if positions:
            text = text[min(positions):]
        row = {**row, "text": text[:6000], "read": True, "fetched_url": markdown_url}
        evidence[source_id] = row
        return row
    tools = {
        "search_web": Tool(schema("search_web", "Search official live documentation sitemap; use query tool calling for this task", {"query": {"type": "string"}}), search_web),
        "read_page": Tool(schema("read_page", "Read a returned source identifier such as L1, not search snippets", {"source_id": {"type": "string", "enum": ["L1", "L2", "L3"]}}), read_page),
    }
    provider = OllamaModel(model, max_output_tokens=384)
    question = "Search official Ollama docs for tool calling using search_web query 'tool calling'. Read at least one returned page using source_id like L1. Summarize the tool call loop in two short sentences. Cite each factual sentence using actual [Lnumber] IDs."
    result = Harness(provider, tools, max_steps=6, max_tools=5, timeout=120).run(question)
    result["agent_loop_status"] = result["status"]
    errors = citation_errors(result["answer"], list(evidence.values())) if evidence else ["no_read_sources"]
    def uncited_lines(answer):
        return any(len(line.strip()) > 25 and not line.strip().startswith(("#", "[L"))
                   and not re.search(r"\[L\d+\]", line) for line in answer.splitlines())
    if evidence and uncited_lines(result["answer"]):
        errors.append("uncited_claim_lines")
    if evidence and errors:
        record = chat([{"role": "system", "content": 'Return JSON {"sentences":[{"text":"one short source-supported statement","source_id":"L1"},{"text":"another statement","source_id":"L1"}]}. Only use actually read source IDs.'},
                       {"role": "user", "content": "Summarize the tool calling loop using only these read sources:\n" + json.dumps(list(evidence.values()))}], model=model, json_mode=True, max_output=256)
        result["raw_answer"] = result["answer"]
        try:
            structured = parse_json(record["response"]["message"]["content"])
            sentences = structured["sentences"][:2]
            result["report_sentence_limit"] = {"proposed": len(structured["sentences"]), "retained": len(sentences)}
            if not sentences or any(s["source_id"] not in evidence or not isinstance(s["text"], str) for s in sentences):
                raise ValueError("invalid report sentence sources")
            result["answer"] = "\n".join(s["text"] + " [" + s["source_id"] + "]" for s in sentences)
        except (ValueError, TypeError, KeyError):
            result["answer"] = record["response"]["message"]["content"]
        result["trace"].append({"event": "citation_repair", "request": record})
        errors = citation_errors(result["answer"], list(evidence.values()))
        if uncited_lines(result["answer"]):
            errors.append("uncited_claim_lines")
    result["evidence"] = list(evidence.values())
    result["citation_errors"] = errors
    result["model"] = model_manifest(model)
    result["model_requests"] = provider.requests
    result["status"] = "completed" if evidence and not errors else "research_incomplete"
    save(output / "result.json", result)
    report = result["answer"] + "\n\n" + "\n".join("["+key+"] "+row["url"] for key, row in evidence.items())
    (output / "report.md").write_text(report, encoding="utf-8")
    print(result["status"], report)
    return result

In [ ]:
evidence=[{"id":"L1","text":"Agent 根据反馈行动"}]
assert citation_errors("结论 [L99]", evidence) == ["L99"]
assert citation_errors("结论 [L1]", evidence) == []
print("编号校验通过仍需核对原文是否支持结论")

In [ ]:
# RUN_LIVE、MODEL 与 RUN_ID 在前面的实验配置格设置；这里复用相同配置。

In [ ]:
if RUN_LIVE:
    web_result = run(OUTPUT / "02-research" / RUN_ID / "web", model=MODEL)
    print("公开资料研究状态：", web_result["status"])
    print("已读来源：", [row["url"] for row in web_result["evidence"]])
    assert web_result["status"] == "completed", web_result
    assert web_result["evidence"] and web_result["citation_errors"] == []
    print("公开资料的工具与引用检查通过；仍需逐句核对摘要是否得到原文支持。")
else:
    print("真实联网研究未执行。启用 RUN_LIVE 后将访问代码中列出的官方公开站点并调用本机模型。")